In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🏦 South African Bank Trust Score System\n",
    "## Helping consumers make informed decisions based on verified data\n",
    "\n",
    "**Data Sources:**\n",
    "- OBS Annual Report 2023 (Ombudsman for Banking Services)\n",
    "- NFO Annual Report 2024 (National Financial Ombud)\n",
    "- SARB Prudential Authority Sanctions 2022-2025\n",
    "- DataEQ South African Banking Sentiment Index 2024\n",
    "- Sagaci Research Consumer Satisfaction 2025\n",
    "\n",
    "**Banks covered:** Standard Bank, FNB, Absa, Nedbank, Capitec, TymeBank\n",
    "\n",
    "---"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# ─────────────────────────────────────────────\n",
    "# IMPORTS\n",
    "# ─────────────────────────────────────────────\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.patches as mpatches\n",
    "import seaborn as sns\n",
    "from matplotlib.colors import LinearSegmentedColormap\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Style\n",
    "plt.rcParams['figure.facecolor'] = '#0d1117'\n",
    "plt.rcParams['axes.facecolor'] = '#161b22'\n",
    "plt.rcParams['axes.labelcolor'] = 'white'\n",
    "plt.rcParams['xtick.color'] = 'white'\n",
    "plt.rcParams['ytick.color'] = 'white'\n",
    "plt.rcParams['text.color'] = 'white'\n",
    "plt.rcParams['axes.titlecolor'] = 'white'\n",
    "plt.rcParams['axes.edgecolor'] = '#30363d'\n",
    "plt.rcParams['grid.color'] = '#30363d'\n",
    "plt.rcParams['font.family'] = 'DejaVu Sans'\n",
    "\n",
    "BANK_COLORS = {\n",
    "    'Standard Bank': '#1565C0',\n",
    "    'FNB': '#E65100',\n",
    "    'Absa': '#C62828',\n",
    "    'Nedbank': '#2E7D32',\n",
    "    'Capitec': '#6A1B9A',\n",
    "    'TymeBank': '#00838F'\n",
    "}\n",
    "\n",
    "print('Libraries loaded successfully')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# ─────────────────────────────────────────────\n",
    "# LOAD DATA\n",
    "# ─────────────────────────────────────────────\n",
    "complaints = pd.read_csv('../data/complaints.csv')\n",
    "sanctions  = pd.read_csv('../data/sanctions.csv')\n",
    "sentiment  = pd.read_csv('../data/sentiment.csv')\n",
    "\n",
    "print('Complaints data:')\n",
    "print(complaints[['bank','formal_cases_2023','referral_conversion_rate_pct','cases_decided_consumer_favour_pct']])\n",
    "print('\\nSanctions data:')\n",
    "print(sanctions[['bank','year','penalty_zar','description']])\n",
    "print('\\nSentiment data:')\n",
    "print(sentiment[['bank','dataeq_net_sentiment_pct','sagaci_satisfaction_2025']])"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Section 1 — Complaint Volume Analysis\n",
    "How many complaints did each bank receive and how has this changed over time?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# ─────────────────────────────────────────────\n",
    "# CHART 1: Complaint volume over 3 years\n",
    "# ─────────────────────────────────────────────\n",
    "fig, ax = plt.subplots(figsize=(12, 6))\n",
    "\n",
    "x = np.arange(len(complaints['bank']))\n",
    "width = 0.25\n",
    "\n",
    "bars1 = ax.bar(x - width, complaints['formal_cases_2021'], width,\n",
    "               label='2021', alpha=0.6, color=[BANK_COLORS[b] for b in complaints['bank']])\n",
    "bars2 = ax.bar(x, complaints['formal_cases_2022'], width,\n",
    "               label='2022', alpha=0.8, color=[BANK_COLORS[b] for b in complaints['bank']])\n",
    "bars3 = ax.bar(x + width, complaints['formal_cases_2023'], width,\n",
    "               label='2023', alpha=1.0, color=[BANK_COLORS[b] for b in complaints['bank']])\n",
    "\n",
    "ax.set_xlabel('Bank', fontsize=12)\n",
    "ax.set_ylabel('Formal Cases Opened', fontsize=12)\n",
    "ax.set_title('Formal Complaints per Bank — 2021 to 2023\\n(Source: OBS Annual Report 2023)', fontsize=14, pad=15)\n",
    "ax.set_xticks(x)\n",
    "ax.set_xticklabels(complaints['bank'], fontsize=11)\n",
    "ax.legend(['2021', '2022', '2023'], facecolor='#161b22', edgecolor='#30363d')\n",
    "ax.grid(axis='y', alpha=0.3)\n",
    "\n",
    "# Add value labels on 2023 bars\n",
    "for bar in bars3:\n",
    "    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,\n",
    "            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/chart_complaints_volume.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "print('Chart saved')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Section 2 — How Well Do Banks Resolve Complaints?\n",
    "The referral conversion rate shows how many complaints escalate to formal cases.\n",
    "A **lower rate is better** — it means the bank resolves complaints before they escalate."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# ─────────────────────────────────────────────\n",
    "# CHART 2: Referral conversion rate\n",
    "# ─────────────────────────────────────────────\n",
    "df_sorted = complaints.sort_values('referral_conversion_rate_pct')\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 6))\n",
    "\n",
    "colors = [BANK_COLORS[b] for b in df_sorted['bank']]\n",
    "bars = ax.barh(df_sorted['bank'], df_sorted['referral_conversion_rate_pct'],\n",
    "               color=colors, edgecolor='#30363d', linewidth=0.5)\n",
    "\n",
    "# Industry average line\n",
    "industry_avg = 52\n",
    "ax.axvline(x=industry_avg, color='#FFD700', linestyle='--', linewidth=1.5,\n",
    "           label=f'Industry Average ({industry_avg}%)')\n",
    "\n",
    "for bar, val in zip(bars, df_sorted['referral_conversion_rate_pct']):\n",
    "    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,\n",
    "            f'{val}%', va='center', fontsize=11, fontweight='bold')\n",
    "\n",
    "ax.set_xlabel('Referral Conversion Rate (%)', fontsize=12)\n",
    "ax.set_title('Referral to Formal Case Conversion Rate\\nLower is better — bank resolved complaint before escalation\\n(Source: OBS Annual Report 2023)',\n",
    "             fontsize=13, pad=15)\n",
    "ax.legend(facecolor='#161b22', edgecolor='#30363d')\n",
    "ax.grid(axis='x', alpha=0.3)\n",
    "ax.set_xlim(0, 90)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/chart_conversion_rate.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Section 3 — Regulatory Sanctions\n",
    "How much has each bank been fined by regulators?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# ─────────────────────────────────────────────\n",
    "# CHART 3: Regulatory sanctions\n",
    "# ─────────────────────────────────────────────\n",
    "sanctions_summary = sanctions.groupby('bank')['penalty_zar'].sum().reset_index()\n",
    "sanctions_summary = sanctions_summary.sort_values('penalty_zar', ascending=False)\n",
    "sanctions_summary['penalty_millions'] = sanctions_summary['penalty_zar'] / 1_000_000\n",
    "\n",
    "# Add banks with no penalties\n",
    "all_banks = pd.DataFrame({'bank': list(BANK_COLORS.keys())})\n",
    "sanctions_summary = all_banks.merge(sanctions_summary, on='bank', how='left').fillna(0)\n",
    "sanctions_summary = sanctions_summary.sort_values('penalty_millions', ascending=True)\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 6))\n",
    "\n",
    "colors = [BANK_COLORS[b] for b in sanctions_summary['bank']]\n",
    "bars = ax.barh(sanctions_summary['bank'], sanctions_summary['penalty_millions'],\n",
    "               color=colors, edgecolor='#30363d')\n",
    "\n",
    "for bar, val in zip(bars, sanctions_summary['penalty_millions']):\n",
    "    if val > 0:\n",
    "        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,\n",
    "                f'R{val:.1f}M', va='center', fontsize=11, fontweight='bold')\n",
    "    else:\n",
    "        ax.text(0.3, bar.get_y() + bar.get_height()/2,\n",
    "                'No financial penalty', va='center', fontsize=10, color='#8b949e')\n",
    "\n",
    "ax.set_xlabel('Total Financial Penalties (ZAR Millions)', fontsize=12)\n",
    "ax.set_title('Total Regulatory Financial Penalties 2022–2025\\n(Source: SARB Prudential Authority)', fontsize=13, pad=15)\n",
    "ax.grid(axis='x', alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/chart_sanctions.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📊 Section 4 — Consumer Sentiment\n",
    "What do consumers actually say about these banks on social media and surveys?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# ─────────────────────────────────────────────\n",
    "# CHART 4: Consumer sentiment\n",
    "# ─────────────────────────────────────────────\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 6))\n",
    "\n",
    "# DataEQ sentiment\n",
    "df_s = sentiment.sort_values('dataeq_net_sentiment_pct')\n",
    "colors = [BANK_COLORS[b] for b in df_s['bank']]\n",
    "axes[0].barh(df_s['bank'], df_s['dataeq_net_sentiment_pct'], color=colors)\n",
    "axes[0].axvline(x=0, color='white', linewidth=0.8)\n",
    "axes[0].set_xlabel('Net Sentiment (%)', fontsize=11)\n",
    "axes[0].set_title('Social Media Net Sentiment\\n(DataEQ 2024 — 3M posts analysed)', fontsize=12)\n",
    "axes[0].grid(axis='x', alpha=0.3)\n",
    "for i, (val, bank) in enumerate(zip(df_s['dataeq_net_sentiment_pct'], df_s['bank'])):\n",
    "    axes[0].text(val + 0.5, i, f'{val}%', va='center', fontsize=10)\n",
    "\n",
    "# Sagaci satisfaction\n",
    "df_sat = sentiment.sort_values('sagaci_satisfaction_2025')\n",
    "colors2 = [BANK_COLORS[b] for b in df_sat['bank']]\n",
    "axes[1].barh(df_sat['bank'], df_sat['sagaci_satisfaction_2025'], color=colors2)\n",
    "axes[1].set_xlabel('Satisfaction Score (%)', fontsize=11)\n",
    "axes[1].set_title('Consumer Satisfaction Score\\n(Sagaci Research 2025)', fontsize=12)\n",
    "axes[1].grid(axis='x', alpha=0.3)\n",
    "for i, val in enumerate(df_sat['sagaci_satisfaction_2025']):\n",
    "    axes[1].text(val + 0.3, i, f'{val}%', va='center', fontsize=10)\n",
    "\n",
    "plt.suptitle('Consumer Sentiment Analysis — South African Banks', fontsize=14, y=1.02)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/chart_sentiment.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 🏆 Section 5 — Trust Score Model\n",
    "Combining all data sources into a single weighted Trust Score per bank.\n",
    "\n",
    "| Dimension | Weight | Source |\n",
    "|---|---|---|\n",
    "| Complaint resolution rate | 30% | OBS 2023 |\n",
    "| Cases decided in consumer's favour | 25% | OBS 2023 + NFO 2024 |\n",
    "| Regulatory sanctions | 25% | SARB Prudential Authority |\n",
    "| Consumer sentiment | 20% | DataEQ + Sagaci |"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# ─────────────────────────────────────────────\n",
    "# TRUST SCORE MODEL\n",
    "# ─────────────────────────────────────────────\n",
    "\n",
    "def normalise(series, invert=False):\n",
    "    \"\"\"Normalise a series to 0-10 scale.\"\"\"\n",
    "    mn, mx = series.min(), series.max()\n",
    "    if mx == mn:\n",
    "        return pd.Series([5.0] * len(series), index=series.index)\n",
    "    normalised = (series - mn) / (mx - mn) * 10\n",
    "    return 10 - normalised if invert else normalised\n",
    "\n",
    "# Merge all datasets\n",
    "df = complaints[['bank','referral_conversion_rate_pct','cases_decided_consumer_favour_pct']].copy()\n",
    "df = df.merge(sentiment[['bank','dataeq_net_sentiment_pct','sagaci_satisfaction_2025']], on='bank')\n",
    "\n",
    "# Sanctions — total penalty per bank\n",
    "sanctions_total = sanctions.groupby('bank')['penalty_zar'].sum().reset_index()\n",
    "sanctions_total.columns = ['bank', 'total_penalty_zar']\n",
    "df = df.merge(sanctions_total, on='bank', how='left').fillna(0)\n",
    "\n",
    "# Score each dimension (0-10)\n",
    "# Lower conversion rate = better score (invert=True)\n",
    "df['score_resolution']  = normalise(df['referral_conversion_rate_pct'], invert=True)\n",
    "# Higher consumer favour = better score\n",
    "df['score_favour']      = normalise(df['cases_decided_consumer_favour_pct'])\n",
    "# Lower penalties = better score (invert=True)\n",
    "df['score_sanctions']   = normalise(df['total_penalty_zar'], invert=True)\n",
    "# Higher sentiment = better score\n",
    "df['score_sentiment']   = normalise(\n",
    "    (df['dataeq_net_sentiment_pct'] + df['sagaci_satisfaction_2025']) / 2\n",
    ")\n",
    "\n",
    "# Weighted Trust Score\n",
    "df['trust_score'] = (\n",
    "    df['score_resolution'] * 0.30 +\n",
    "    df['score_favour']     * 0.25 +\n",
    "    df['score_sanctions']  * 0.25 +\n",
    "    df['score_sentiment']  * 0.20\n",
    ")\n",
    "\n",
    "df = df.sort_values('trust_score', ascending=False)\n",
    "\n",
    "print('\\n=== SOUTH AFRICAN BANK TRUST SCORES ===')\n",
    "print(f'{\"Bank\":<15} {\"Resolution\":>12} {\"Favour\":>8} {\"Sanctions\":>11} {\"Sentiment\":>11} {\"TRUST SCORE\":>13}')\n",
    "print('-' * 75)\n",
    "for _, row in df.iterrows():\n",
    "    print(f\"{row['bank']:<15} {row['score_resolution']:>12.1f} {row['score_favour']:>8.1f} {row['score_sanctions']:>11.1f} {row['score_sentiment']:>11.1f} {row['trust_score']:>13.1f}/10\")\n",
    "print('=' * 75)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# ─────────────────────────────────────────────\n",
    "# CHART 5: Final Trust Score\n",
    "# ─────────────────────────────────────────────\n",
    "fig, ax = plt.subplots(figsize=(12, 7))\n",
    "\n",
    "df_plot = df.sort_values('trust_score')\n",
    "colors = [BANK_COLORS[b] for b in df_plot['bank']]\n",
    "\n",
    "bars = ax.barh(df_plot['bank'], df_plot['trust_score'],\n",
    "               color=colors, edgecolor='#30363d', height=0.6)\n",
    "\n",
    "# Add score labels\n",
    "for bar, val in zip(bars, df_plot['trust_score']):\n",
    "    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,\n",
    "            f'{val:.1f}/10', va='center', fontsize=13, fontweight='bold')\n",
    "\n",
    "# Colour zones\n",
    "ax.axvspan(0, 4, alpha=0.05, color='red', label='Low Trust')\n",
    "ax.axvspan(4, 7, alpha=0.05, color='yellow', label='Medium Trust')\n",
    "ax.axvspan(7, 10, alpha=0.05, color='green', label='High Trust')\n",
    "\n",
    "ax.set_xlabel('Trust Score (out of 10)', fontsize=13)\n",
    "ax.set_title('🏦 South African Bank Trust Score\\nBased on complaints, regulatory sanctions and consumer sentiment',\n",
    "             fontsize=14, pad=15)\n",
    "ax.set_xlim(0, 11)\n",
    "ax.grid(axis='x', alpha=0.3)\n",
    "ax.legend(facecolor='#161b22', edgecolor='#30363d', loc='lower right')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/chart_trust_score.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# ─────────────────────────────────────────────\n",
    "# CHART 6: Radar chart — all dimensions\n",
    "# ─────────────────────────────────────────────\n",
    "from matplotlib.patches import FancyArrowPatch\n",
    "import matplotlib.gridspec as gridspec\n",
    "\n",
    "categories = ['Complaint\\nResolution', 'Consumer\\nFavour', 'Regulatory\\nRecord', 'Public\\nSentiment']\n",
    "N = len(categories)\n",
    "angles = [n / float(N) * 2 * np.pi for n in range(N)]\n",
    "angles += angles[:1]\n",
    "\n",
    "fig, axes = plt.subplots(2, 3, figsize=(16, 10),\n",
    "                          subplot_kw=dict(projection='polar'))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for idx, (_, row) in enumerate(df.iterrows()):\n",
    "    values = [\n",
    "        row['score_resolution'],\n",
    "        row['score_favour'],\n",
    "        row['score_sanctions'],\n",
    "        row['score_sentiment']\n",
    "    ]\n",
    "    values += values[:1]\n",
    "\n",
    "    ax = axes[idx]\n",
    "    color = BANK_COLORS[row['bank']]\n",
    "\n",
    "    ax.plot(angles, values, 'o-', linewidth=2, color=color)\n",
    "    ax.fill(angles, values, alpha=0.25, color=color)\n",
    "    ax.set_xticks(angles[:-1])\n",
    "    ax.set_xticklabels(categories, size=9)\n",
    "    ax.set_ylim(0, 10)\n",
    "    ax.set_yticks([2, 4, 6, 8, 10])\n",
    "    ax.set_yticklabels(['2', '4', '6', '8', '10'], size=7, color='#8b949e')\n",
    "    ax.grid(color='#30363d', linewidth=0.5)\n",
    "    ax.set_facecolor('#161b22')\n",
    "    ax.set_title(f\"{row['bank']}\\nTrust Score: {row['trust_score']:.1f}/10\",\n",
    "                 size=11, fontweight='bold', color=color, pad=15)\n",
    "\n",
    "fig.patch.set_facecolor('#0d1117')\n",
    "plt.suptitle('Bank Performance Radar — All Dimensions', fontsize=15, y=1.02)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/chart_radar.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 📋 Section 6 — Consumer Decision Guide\n",
    "Plain language summary to help consumers choose a bank."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# ─────────────────────────────────────────────\n",
    "# CONSUMER DECISION GUIDE\n",
    "# ─────────────────────────────────────────────\n",
    "def get_trust_label(score):\n",
    "    if score >= 7: return '🟢 HIGH TRUST'\n",
    "    elif score >= 4: return '🟡 MEDIUM TRUST'\n",
    "    else: return '🔴 LOW TRUST'\n",
    "\n",
    "def get_recommendation(row):\n",
    "    recs = []\n",
    "    if row['referral_conversion_rate_pct'] <= 40:\n",
    "        recs.append('✅ Resolves complaints quickly before escalation')\n",
    "    else:\n",
    "        recs.append('⚠️  High complaint escalation rate — may be slow to resolve issues')\n",
    "    if row['cases_decided_consumer_favour_pct'] >= 75:\n",
    "        recs.append('✅ High rate of cases decided in consumer favour')\n",
    "    else:\n",
    "        recs.append('⚠️  Lower rate of cases decided in consumer favour')\n",
    "    if row['total_penalty_zar'] == 0:\n",
    "        recs.append('✅ No financial regulatory penalties on record')\n",
    "    elif row['total_penalty_zar'] < 20_000_000:\n",
    "        recs.append(f\"⚠️  R{row['total_penalty_zar']/1_000_000:.0f}M in regulatory penalties\")\n",
    "    else:\n",
    "        recs.append(f\"🔴 R{row['total_penalty_zar']/1_000_000:.0f}M in regulatory penalties\")\n",
    "    if row['sagaci_satisfaction_2025'] >= 55:\n",
    "        recs.append('✅ Above average consumer satisfaction')\n",
    "    else:\n",
    "        recs.append('⚠️  Below average consumer satisfaction')\n",
    "    return recs\n",
    "\n",
    "print('\\n' + '='*65)\n",
    "print('  SOUTH AFRICAN BANK TRUST GUIDE FOR CONSUMERS')\n",
    "print('  Based on verified data from NFO, SARB, DataEQ & Sagaci')\n",
    "print('='*65)\n",
    "\n",
    "for _, row in df.iterrows():\n",
    "    print(f\"\\n🏦 {row['bank']}\")\n",
    "    print(f\"   Trust Score: {row['trust_score']:.1f}/10 — {get_trust_label(row['trust_score'])}\")\n",
    "    for rec in get_recommendation(row):\n",
    "        print(f\"   {rec}\")\n",
    "\n",
    "print('\\n' + '='*65)\n",
    "print('  DATA SOURCES')\n",
    "print('  OBS Annual Report 2023 | NFO Annual Report 2024')\n",
    "print('  SARB Prudential Authority 2022-2025')\n",
    "print('  DataEQ SA Banking Index 2024 | Sagaci Research 2025')\n",
    "print('='*65)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}